In [ ]:
from pyspark.sql import functions as F

**Loading Data**

In [ ]:
users_raw = spark.read.table("raw_users")

campaigns_raw = spark.read.table("raw_campaigns")

ad_requests_raw = spark.read.table("raw_ad_requests")

In [ ]:
print("Users:", users_raw.count())
print("Campaigns:", campaigns_raw.count())
print("Ad Requests:", ad_requests_raw.count())

**Users Cleaning** 

1.1 Displaying data

In [ ]:
print("Raw rows:", users_raw.count())

users_raw.printSchema()

display(users_raw)

1.2 Remove exact duplicate rows

In [ ]:
users_clean = users_raw.dropDuplicates()

print("Raw rows:", users_raw.count())
print("Rows after duplicate removal:", users_clean.count())

1.3 Missing Username and Email

In [ ]:
##keeping them as null

1.4 Standardize role

In [ ]:
users_clean = users_clean.withColumn(
    "role",
    F.trim(F.col("role"))
)

users_clean = users_clean.withColumn(
    "role",
    F.lower(F.col("role"))
)

1.5 Validation step

In [ ]:
#roles
invalid_roles = users_clean.filter(
    ~F.col("role").isin("influencer", "sponsor", "admin")
)

print("Invalid role rows:", invalid_roles.count())

display(invalid_roles)

#duplicates
duplicate_user_ids = users_clean.groupBy("id") \
    .count() \
    .filter(F.col("count") > 1)

print("Duplicate UserID values:", duplicate_user_ids.count())

display(duplicate_user_ids)

In [ ]:
users_clean.select(
    F.count("*").alias("total_rows"),
    F.count(F.when(F.col("id").isNull(), 1)).alias("missing_id"),
    F.count(F.when(F.col("username").isNull(), 1)).alias("missing_username"),
    F.count(F.when(F.col("email").isNull(), 1)).alias("missing_email"),
    F.count(F.when(F.col("role").isNull(), 1)).alias("missing_role")
).show()

users_clean.groupBy("role") \
    .count() \
    .show()

**Campaigns Cleaning**

2.1
Loading user data

In [ ]:
#already loaded, doing further steps
print("Raw rows:", campaigns_raw.count())

campaigns_raw.printSchema()

display(campaigns_raw)

2.2 Remove exact duplicate rows

In [ ]:
campaigns_clean = campaigns_raw.dropDuplicates()

print("Raw rows:", campaigns_raw.count())
print("Rows after duplicate removal:", campaigns_clean.count())

2.3 Missing Description and Budget

In [ ]:
#process to check missing values
campaigns_clean.select(
    F.count(F.when(F.col("description").isNull(), 1))
        .alias("missing_description"),

    F.count(F.when(F.col("budget").isNull(), 1))
        .alias("missing_budget")
).show()

# we will keep them as null only

2.4 Standardize visibility

In [ ]:
campaigns_clean = campaigns_clean.withColumn(
    "visibility",
    F.trim(F.col("visibility"))
)

campaigns_clean = campaigns_clean.withColumn(
    "visibility",
    F.lower(F.col("visibility"))
)

2.5 Create invalid_budget_flag and convert negatives to NULL

In [ ]:
#jsut checking again
negative_budgets = campaigns_clean.filter(
    F.col("budget") < 0
)

print("Negative budget rows:", negative_budgets.count())

display(
    negative_budgets.select(
        "id",
        "budget"
    )
)


In [ ]:
#flagging
campaigns_clean = campaigns_clean.withColumn(
    "invalid_budget_flag",
    F.when(F.col("budget")<0,1).otherwise(0)
)

In [ ]:
#changing bugdet<0 to nulls
campaigns_clean = campaigns_clean.withColumn(
    "budget",
    F.when(
        F.col("budget")<0,None
    ).otherwise(F.col("budget"))
)

2.6 Budget Outliers (creating flag)

In [ ]:
#Calculating 
budget_stats = campaigns_clean.select(
    F.expr("percentile_approx(budget, 0.25)").alias("Q1"),
    F.expr("percentile_approx(budget, 0.75)").alias("Q3")
).collect()[0]

q1 = budget_stats["Q1"]
q3 = budget_stats["Q3"]

iqr = q3 - q1
upper_limit = q3 + (1.5 * iqr)

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Upper outlier limit:", upper_limit)

In [ ]:
#flagging
campaigns_clean = campaigns_clean.withColumn(
    "budget_outlier_flag",
    F.when(
        F.col("budget") > upper_limit,
        1
    ).otherwise(0)
)

2.7 Standardize dates

In [ ]:
#standardized date columns
campaigns_clean = campaigns_clean.withColumn(
    "start_date_clean",
    F.coalesce(
        F.to_date("start_date", "yyyy-MM-dd"),
        F.to_date("start_date", "dd-MM-yyyy"),
        F.to_date("start_date", "MM/dd/yyyy"),
        F.to_date("start_date", "yyyy/MM/dd"),
        F.to_date("start_date", "MM-dd-yyyy"),
        F.to_date("start_date", "dd/MM/yyyy")
    )
)

campaigns_clean = campaigns_clean.withColumn(
    "end_date_clean",
    F.coalesce(
        F.to_date("end_date", "yyyy-MM-dd"),
        F.to_date("end_date", "dd-MM-yyyy"),
        F.to_date("end_date", "MM/dd/yyyy"),
        F.to_date("end_date", "yyyy/MM/dd"),
        F.to_date("end_date", "MM-dd-yyyy"),
        F.to_date("end_date", "dd/MM/yyyy")
    )
)

**open and see**

              Raw date
                  ↓
        ┌─────────┴─────────┐
        ↓         ↓         ↓
   yyyy-MM-dd dd-MM-yyyy MM/dd/yyyy
        ↓         ↓         ↓
      NULL   2025-03-15    NULL
                  ↓
            COALESCE()
                  ↓
           2025-03-15
--more types are also added

In [ ]:
#Handle unparseable dates and creating flags
#to identify missing and invalid dates
campaigns_clean = campaigns_clean.withColumn(
    "invalid_start_date_flag",
    F.when(
        F.col("start_date_clean").isNull() &
        F.col("start_date").isNotNull(),
        1
    ).otherwise(0)
)

campaigns_clean = campaigns_clean.withColumn(
    "invalid_end_date_flag",
    F.when(
        F.col("end_date_clean").isNull() &
        F.col("end_date").isNotNull(),
        1
    ).otherwise(0)
)

In [ ]:
#invalid start and end dates
campaigns_clean.select(
    F.sum("invalid_start_date_flag").alias("invalid_start_dates"),
    F.sum("invalid_end_date_flag").alias("invalid_end_dates")
).show()

2.8 Detect end date before start date and flag

In [ ]:
#checking for invalid date range
campaigns_clean = campaigns_clean.withColumn(
    "invalid_date_range_flag",
    F.when(
        (F.col("start_date_clean").isNotNull()) &
        (F.col("end_date_clean").isNotNull()) &
        (F.col("end_date_clean") < F.col("start_date_clean")),
        1
    ).otherwise(0)
)

In [ ]:
#invalid dates
campaigns_clean.select(
    F.sum("invalid_date_range_flag").alias("invalid_date_ranges")
).show()

In [ ]:
#now we can replace the original string date 
#columns with the cleaned date columns
#not needed for other cols, as they were directly changed, but here we wanted
#to preserve the original date temporarily while testing multiple formats.
campaigns_clean = campaigns_clean \
    .drop("start_date", "end_date") \
    .withColumnRenamed("start_date_clean", "start_date") \
    .withColumnRenamed("end_date_clean", "end_date")

2.9 Validation

In [ ]:
#Validate budget outliers
campaigns_clean.groupBy("budget_outlier_flag") \
    .count() \
    .show()

#Outliers are kept.
#We're only flagging them because:
#statistically unusual ≠ automatically invalid.

In [ ]:
#validate visibility
campaigns_clean.groupBy("visibility") \
    .count() \
    .orderBy("visibility") \
    .show()

In [ ]:
#Final Campaign validation
campaigns_clean.select(
    F.count("*").alias("total_rows"),

    F.count(F.when(F.col("id").isNull(), 1))
        .alias("missing_id"),

    F.count(F.when(F.col("description").isNull(), 1))
        .alias("missing_description"),

    F.count(F.when(F.col("budget").isNull(), 1))
        .alias("null_budget"),

    F.sum("invalid_budget_flag")
        .alias("invalid_budget"),

    F.sum("budget_outlier_flag")
        .alias("budget_outliers"),

    F.sum("invalid_start_date_flag")
        .alias("invalid_start_dates"),

    F.sum("invalid_end_date_flag")
        .alias("invalid_end_dates"),

    F.sum("invalid_date_range_flag")
        .alias("invalid_date_ranges")
).show()

**AD Requests**

In [ ]:
#already loaded

3.1 Remove exact duplicate rows

In [ ]:
ad_requests_clean = ad_requests_raw.dropDuplicates()
print("Rows with duplicates:", ad_requests_raw.count())
print("Rows after removing duplicates:", ad_requests_clean.count())

3.2 Check the missing values we decided to keep

In [ ]:
ad_requests_clean.select(
    F.count(F.when(F.col("messages").isNull(), 1)).alias("missing_messages"),
    F.count(F.when(F.col("payment_amount").isNull(), 1)).alias("missing_payments"),
    F.count(F.when(F.col("status").isNull(), 1)).alias("missing_status")
).show()
#we will keep them as null only

3.3 Standardize status formatting

In [ ]:
ad_requests_clean = ad_requests_clean.withColumn(
    "status",
    F.trim(F.col("status"))
)

ad_requests_clean = ad_requests_clean.withColumn(
    "status",
    F.lower(F.col("status"))
)

3.4 Standardize known spelling mistakes

In [ ]:
#spelling mistakes -> these were present in ads only
ad_requests_clean = ad_requests_clean.withColumn(
    "status",
    F.when(F.col("status") == "accpeted", "accepted")
     .when(F.col("status") == "compeleted", "completed")
     .when(F.col("status") == "negotaiate", "negotiate")
     .when(F.col("status") == "pendng", "pending")
     .otherwise(F.col("status"))
)

3.5 Creating the missing-status flag

In [ ]:
ad_requests_clean = ad_requests_clean.withColumn(
    "missing_status_flag",
    F.when(F.col("status").isNull(), 1).otherwise(0)
)

3.6 Handle negative payment amounts

In [ ]:
#creating flag
ad_requests_clean = ad_requests_clean.withColumn(
    "invalid_payment_flag",
    F.when(F.col("payment_amount") < 0, 1).otherwise(0)
)

In [ ]:
#converting negative to null
ad_requests_clean = ad_requests_clean.withColumn(
    "payment_amount",
    F.when(
        F.col("payment_amount") < 0,
        None
    ).otherwise(F.col("payment_amount"))
)

3.7 Flag Payment Outliers

In [ ]:
#outliers
payment_stats = ad_requests_clean.select(
    F.expr("percentile_approx(payment_amount, 0.25)").alias("Q1"),
    F.expr("percentile_approx(payment_amount, 0.75)").alias("Q3")
).collect()[0]

q1 = payment_stats["Q1"]
q3 = payment_stats["Q3"]

iqr = q3 - q1
upper_limit = q3 + (1.5 * iqr)

print("Q1:", q1)
print("Q3:", q3)
print("Upper limit:", upper_limit)

In [ ]:
#flag
ad_requests_clean = ad_requests_clean.withColumn(
    "payment_outlier_flag",
    F.when(
        F.col("payment_amount") > upper_limit,
        1
    ).otherwise(0)
)

3.8 Validating

In [ ]:
#Validate the payment cleaning
ad_requests_clean.select(
    F.sum("invalid_payment_flag").alias("invalid_payments"),
    F.sum("payment_outlier_flag").alias("payment_outliers")
).show()
#You should see the number of negative payments flagged, 
#and the number of statistical outliers identified.

In [ ]:
#no negative payment remained
ad_requests_clean.filter(
    F.col("payment_amount") < 0
).count()

In [ ]:
#NULL payment count(invalid payment converted to null will be added here)
ad_requests_clean.select(
    F.count(F.when(F.col("payment_amount").isNull(), 1))
    .alias("null_payments")
).show()

In [ ]:
#we're checking both existence + role(campaignid,infuencerid,sponsorid)
#valid campaign IDs
valid_campaign_ids = campaigns_clean.select(
    F.col("id").alias("campaign_id")
).distinct()

#Ad Requests whose campaign ID doesn't exist
invalid_campaign_ids = ad_requests_clean.join(
    valid_campaign_ids,
    on="campaign_id",
    how="left_anti"
)

#checking
print("Invalid campaign ID rows:", invalid_campaign_ids.count())
display(
    invalid_campaign_ids.select(
        "id",
        "campaign_id"
    )
)

In [ ]:
#Validate Influencer IDs
valid_influencer_ids = users_clean.filter(
    F.col("role") == "influencer"
).select(
    F.col("id").alias("influencer_id")
).distinct()

invalid_influencer_ids = ad_requests_clean.join(
    valid_influencer_ids,
    on="influencer_id",
    how="left_anti"
)

print(
    "Invalid influencer ID rows:",
    invalid_influencer_ids.count()
)

display(
    invalid_influencer_ids.select(
        "id",
        "influencer_id"
    )
)

In [ ]:
#flagging invalid influencer id
ad_requests_clean = ad_requests_clean.withColumn(
    "invalid_influencer_id_flag",
    F.when(
        F.col("influencer_id").isin(20001, 20002, 20003),
        1
    ).otherwise(0)
)

In [ ]:
#Validate Sponsor IDs
valid_sponsor_ids = users_clean.filter(
    F.col("role") == "sponsor"
).select(
    F.col("id").alias("sponsor_id")
).distinct()

invalid_sponsor_ids = ad_requests_clean.join(
    valid_sponsor_ids,
    on="sponsor_id",
    how="left_anti"
)

print(
    "Invalid sponsor ID rows:",
    invalid_sponsor_ids.count()
)

display(
    invalid_sponsor_ids.select(
        "id",
        "sponsor_id"
    )
)

In [ ]:
#flagging invalid sponsor id
ad_requests_clean = ad_requests_clean.withColumn(
    "invalid_sponsor_id_flag",
    F.when(
        F.col("sponsor_id").isin(30001, 30002, 30003),
        1
    ).otherwise(0)
)

In [ ]:
#verifying
ad_requests_clean.select(
    F.sum("invalid_influencer_id_flag").alias("invalid_influencer_ids"),
    F.sum("invalid_sponsor_id_flag").alias("invalid_sponsor_ids")
).show()

In [ ]:
#Validate Status Values
valid_statuses = [
    "pending",
    "accepted",
    "negotiate",
    "completed",
    "paid",
    "new"
]

invalid_statuses = ad_requests_clean.filter(
    F.col("status").isNotNull() &
    ~F.col("status").isin(valid_statuses)
)

print(
    "Invalid status rows:",
    invalid_statuses.count()
)

display(
    invalid_statuses.select(
        "id",
        "status"
    )
)

In [ ]:
#Final Ad Requests Validation
ad_requests_clean.select(
    F.count("*").alias("total_rows"),

    F.count(
        F.when(F.col("id").isNull(), 1)
    ).alias("missing_id"),

    F.count(
        F.when(F.col("messages").isNull(), 1)
    ).alias("missing_messages"),

    F.count(
        F.when(F.col("payment_amount").isNull(), 1)
    ).alias("null_payments"),

    F.sum("missing_status_flag").alias("missing_status"),

    F.sum("invalid_payment_flag").alias("invalid_payments"),

    F.sum("payment_outlier_flag").alias("payment_outliers")
).show()

In [ ]:
#validating foreign keys
print("Invalid campaign IDs:", invalid_campaign_ids.count())
print("Invalid influencer IDs:", invalid_influencer_ids.count())
print("Invalid sponsor IDs:", invalid_sponsor_ids.count())
print("Invalid status values:", invalid_statuses.count())

**Building silver dataset(storing cleaned files in delta lake)**

4.1 Creating Silver users

In [ ]:
#just verifying once
print("Users rows:", users_clean.count())
users_clean.printSchema()
display(users_clean)

In [ ]:
#writing users as delta table
users_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_users")

# overwrite means: If silver_users already exists, replace it.
#overwriteSchema = true tells Delta Lake to replace the existing table's 
#schema with the schema of your new DataFrame.

In [ ]:
spark.sql("""
    SELECT *
    FROM silver_users
    LIMIT 10
""").show()

print(
    "Silver Users rows:",
    spark.table("silver_users").count()
)

4.2 Creating Silver campaigns

In [ ]:
#delta table Campaigns
campaigns_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_campaigns")

In [ ]:
#validating
spark.sql("""
    SELECT *
    FROM silver_campaigns
    LIMIT 10
""").show()

print(
    "Silver Campaigns rows:",
    spark.table("silver_campaigns").count()
)


4.3 Creating Silver Ad requests

In [ ]:
#delta table Ad Requests
ad_requests_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_ad_requests")

In [ ]:
#validating
spark.sql("""
    SELECT *
    FROM silver_ad_requests
    LIMIT 10
""").show()

print(
    "Silver Ad Requests rows:",
    spark.table("silver_ad_requests").count()
)

4.4 Final Validation

In [ ]:
print("Silver Users:", spark.table("silver_users").count())
print("Silver Campaigns:", spark.table("silver_campaigns").count())
print("Silver Ad Requests:", spark.table("silver_ad_requests").count())

**Building Star Schema**

click and see schema design
                    dim_user
                       │
                       │
                       ▼
dim_campaign ─── fact_ad_request
                       │
                       │
                       ▼
                    dim_date

5.1 Create dim_user

In [ ]:
#loading silver table
users_silver = spark.table("silver_users")

In [ ]:
# data
display(users_silver)

In [ ]:
#we dont need password hash, thus we are deleting it and creating dim_user
dim_user = users_silver.select(
    F.col("id").alias("user_id"),
    "username",
    "email",
    "role"
)

In [ ]:
#validating
display(dim_user)
print(dim_user.count())

#finally
dim_user
─────────────────────────────
user_id        ← user
username       ← user name
email          ← contact
role           ← influencer/sponsor/admin

5.2 Create dim_campaign

In [ ]:
#loading
campaigns_silver = spark.table("silver_campaigns")

In [ ]:
#data 
display(campaigns_silver)

In [ ]:
#For the Gold dimension, we don't need to carry every technical cleaning column.
#thus we remove invalid_budget_flag, budget_outlier_flag ,invalid_start_date_flag
#invalid_end_date_flag,invalid_date_range_flag
dim_campaigns = campaigns_silver.select(
    F.col("id").alias("campaign_id"),
    "name",
    "description",
    "budget",
    "visibility",
    "start_date",
    "end_date"
)

In [ ]:
#validating
display(dim_campaigns)
print("dim_campaign rows:", dim_campaigns.count())

#finally
dim_campaign
─────────────────────────────
campaign_id    ← campaign
name           ← campaign name
description    ← campaign details
budget         ← campaign budget
visibility     ← public/private
start_date     ← campaign start
end_date       ← campaign end

5.3 Create dim_date

In [ ]:
from pyspark.sql import functions as F

In [ ]:
#we are generating all the dates b/w max and min dates
#Because dim_date is a calendar dimension, not a table containing 
#only dates where something happened.
#useful for time based anl,we getdates with zero act and consistent time attributes
# Generate dim_date dynamically from campaign dates

campaign_date_range = campaigns_silver.select(
    F.min("start_date").alias("min_date"),
    F.max("end_date").alias("max_date")
).collect()[0]

date_df = campaigns_silver.select(
    F.explode(
        F.sequence(
            F.lit(campaign_date_range["min_date"]),
            F.lit(campaign_date_range["max_date"]),
            F.expr("interval 1 day")
        )
    ).alias("date")
).distinct()

In [ ]:
# adding useful date attributes
dim_date = date_df.select(
    F.date_format("date", "yyyyMMdd").cast("int").alias("date_id"),
    "date",
    F.day("date").alias("day"),
    F.month("date").alias("month"),
    F.date_format("date", "MMMM").alias("month_name"),
    F.quarter("date").alias("quarter"),
    F.year("date").alias("year"),
    F.dayofweek("date").alias("day_of_week"),
    F.date_format("date", "EEEE").alias("day_name")
)

In [ ]:
display(dim_date)

#finally
dim_date
─────────────────────────────
date_id        ← date key
date           ← calendar date
day            ← day number
month          ← month number
month_name     ← month name
quarter        ← quarter
year           ← year
day_of_week    ← weekday number
day_name       ← weekday name

5.4 Create fact_ad_request

In [ ]:
#loading
ad_requests_silver = spark.table("silver_ad_requests")

In [ ]:
display(ad_requests_silver)

In [ ]:
# Building Gold fact table
ar = ad_requests_silver.alias("ar")

c = dim_campaigns.select(
    "campaign_id",
    "start_date"
).alias("c")

d = dim_date.select(
    "date_id",
    F.col("date").alias("campaign_date")
).alias("d")

# Valid users only
u = dim_user.select(
    F.col("user_id").alias("valid_user_id")
).distinct().alias("u")

fact_ad_request = (
    ar
    # Campaign relationship
    .join(
        c,
        F.col("ar.campaign_id") == F.col("c.campaign_id"),
        "inner"
    )

    # Influencer must exist in dim_user
    .join(
        u.alias("inf"),
        F.col("ar.influencer_id") == F.col("inf.valid_user_id"),
        "inner"
    )

    # Sponsor must exist in dim_user
    .join(
        u.alias("sp"),
        F.col("ar.sponsor_id") == F.col("sp.valid_user_id"),
        "inner"
    )

    # Campaign start date → date dimension
    .join(
        d,
        F.col("c.start_date") == F.col("d.campaign_date"),
        "inner"
    )

    .select(
        F.col("ar.id").alias("request_id"),
        F.col("ar.campaign_id"),
        F.col("ar.influencer_id"),
        F.col("ar.sponsor_id"),
        F.col("d.date_id"),
        F.col("ar.payment_amount"),
        F.col("ar.status")
    )
)

#virtually
             silver_campaigns
                    │
                    │ campaign_id
                    │
silver_ad_requests ─┘
        │
        │ start_date
        ▼
    dim_date
        │
        │ date_id
        ▼
fact_ad_request

In [ ]:
display(fact_ad_request)
print(fact_ad_request.count())

#finally
fact_ad_request
─────────────────────────────
request_id       ← request
campaign_id      ← campaign
influencer_id    ← user
sponsor_id       ← user
date_id          ← date
payment_amount   ← measure
status           ← attribute

5.5 Validation

In [ ]:
#campaign_id
invalid_fact_campaigns = fact_ad_request.join(
    dim_campaigns.select("campaign_id").distinct(),
    on="campaign_id",
    how="left_anti"
)

print("Invalid campaign IDs in fact:", invalid_fact_campaigns.count())
#it is coming 0, which is what we need

In [ ]:
#influencer_id
invalid_fact_influencers = fact_ad_request.join(
    dim_user.select("user_id").distinct(),
    fact_ad_request.influencer_id == dim_user.user_id,
    how="left_anti"
)

print("Invalid influencer IDs in fact:", invalid_fact_influencers.count())
#it is coming 0, which is what we need

In [ ]:
#sponsor_id
invalid_fact_sponsors = fact_ad_request.join(
    dim_user.select("user_id").distinct(),
    fact_ad_request.sponsor_id == dim_user.user_id,
    how="left_anti"
)

print("Invalid sponsor IDs in fact:", invalid_fact_sponsors.count())


In [ ]:
#date_id
invalid_fact_dates = fact_ad_request.join(
    dim_date.select("date_id").distinct(),
    on="date_id",
    how="left_anti"
)

print("Invalid date IDs in fact:", invalid_fact_dates.count())

print(
    "NULL date IDs in fact:",
    fact_ad_request.filter(F.col("date_id").isNull()).count()
)

**Writing dim and fact table to delta table**

6.1 dim_user

In [ ]:
dim_user.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_user")

6.2 dim_campaign

In [ ]:
dim_campaigns.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_campaigns")

6.3 dim_date

In [ ]:
dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_date")

6.4 fact_ad_request

In [ ]:
fact_ad_request.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fact_ad_request")

6.5 final validation

In [ ]:
gold_fact = spark.table("fact_ad_request")
gold_users = spark.table("dim_user")
gold_campaigns = spark.table("dim_campaigns")
gold_dates = spark.table("dim_date")

In [ ]:
#campaigns
print(
    "Invalid campaign relationships:",
    gold_fact.join(
        gold_campaigns.select("campaign_id").distinct(),
        on="campaign_id",
        how="left_anti"
    ).count()
)

In [ ]:
#influencer
print(
    "Invalid influencer relationships:",
    gold_fact.join(
        gold_users.select("user_id").distinct(),
        gold_fact.influencer_id == gold_users.user_id,
        "left_anti"
    ).count()
)

In [ ]:
#sponsor
print(
    "Invalid sponsor relationships:",
    gold_fact.join(
        gold_users.select("user_id").distinct(),
        gold_fact.sponsor_id == gold_users.user_id,
        "left_anti"
    ).count()
)

In [ ]:
#date
print(
    "Invalid date relationships:",
    gold_fact.join(
        gold_dates.select("date_id").distinct(),
        on="date_id",
        how="left_anti"
    ).count()
)

In [ ]:
#foreign key
gold_fact.select(
    F.count(F.when(F.col("campaign_id").isNull(), 1)).alias("null_campaign_id"),
    F.count(F.when(F.col("influencer_id").isNull(), 1)).alias("null_influencer_id"),
    F.count(F.when(F.col("sponsor_id").isNull(), 1)).alias("null_sponsor_id"),
    F.count(F.when(F.col("date_id").isNull(), 1)).alias("null_date_id")
).show()

all 0 as expected

****

**⭐ Reload Tables for Analytics**

In [ ]:
from pyspark.sql import functions as F

users_silver = spark.table("silver_users")
campaigns_silver = spark.table("silver_campaigns")
ad_requests_silver = spark.table("silver_ad_requests")

dim_user = spark.table("dim_user")
dim_campaigns = spark.table("dim_campaigns")
dim_date = spark.table("dim_date")
fact = spark.table("fact_ad_request")

**Gold Analytics**

5.1 basic analytics

In [ ]:
#1) total no of ad request
fact = spark.table("fact_ad_request")

fact.select(
    F.count("*").alias("total_requests")
).show()

In [ ]:
#2) Total payment
fact.select(
    F.sum("payment_amount").alias("total_payment")
).show()

In [ ]:
#3) Requests by status
fact.groupBy("status")\
    .count()\
    .orderBy(F.desc("count"))\
    .show()

5.2 Campaign Analytics

In [ ]:
#How many ad requests does each campaign have, 
#and how much payment is associated with each campaign?
campaign_analytics = fact.alias("f")\
    .join(
        spark.table("dim_campaigns").alias("c"),
        F.col("f.campaign_id") == F.col("c.campaign_id"),
        "left"
    )\
    .groupBy(
        "c.campaign_id",
        "c.name"
    )\
    .agg(
        F.count("f.request_id").alias("request_count"),
        F.sum("f.payment_amount").alias("total_payment")
    ) \
    .orderBy(F.desc("request_count"))

display(campaign_analytics)

5.3 Influencer Analytics

In [ ]:
#How many ad requests did each influencer receive, and how much did they earn?
influencer_analytics = fact.alias("f") \
    .join(
        spark.table("dim_user").alias("u"),
        F.col("f.influencer_id") == F.col("u.user_id"),
        "left"
    ) \
    .filter(F.col("u.role") == "influencer") \
    .groupBy(
        "u.user_id",
        "u.username"
    ) \
    .agg(
        F.count("f.request_id").alias("request_count"),
        F.sum("f.payment_amount").alias("total_earnings")
    ) \
    .orderBy(F.desc("total_earnings"))

display(influencer_analytics)

5.4 Sponsor Analytics

In [ ]:
#How many ad requests did each sponsor make, and what is the
#total payment associated with those requests?
sponsor_analytics = fact.alias("f") \
    .join(
        spark.table("dim_user").alias("u"),
        F.col("f.sponsor_id") == F.col("u.user_id"),
        "left"
    ) \
    .filter(F.col("u.role") == "sponsor") \
    .groupBy(
        "u.user_id",
        "u.username"
    ) \
    .agg(
        F.count("f.request_id").alias("request_count"),
        F.sum("f.payment_amount").alias("total_payment")
    ) \
    .orderBy(F.desc("request_count"))

display(sponsor_analytics)

5.5 Date Analytics

In [ ]:
#How many ad requests and how much payment occurred over time?
monthly_analytics = fact.alias("f") \
    .join(
        spark.table("dim_date").alias("d"),
        F.col("f.date_id") == F.col("d.date_id"),
        "left"
    ) \
    .groupBy(
        "d.year",
        "d.month",
        "d.month_name"
    ) \
    .agg(
        F.count("f.request_id").alias("request_count"),
        F.sum("f.payment_amount").alias("total_payment")
    ) \
    .orderBy(
        "d.year",
        "d.month"
    )

display(monthly_analytics)

5.6 Campaign performance by status

In [ ]:
campaign_status_analytics = fact.alias("f") \
    .join(
        spark.table("dim_campaigns").alias("c"),
        F.col("f.campaign_id") == F.col("c.campaign_id"),
        "left"
    ) \
    .groupBy(
        "c.campaign_id",
        "c.name",
        "f.status"
    ) \
    .agg(
        F.count("f.request_id").alias("request_count"),
        F.sum("f.payment_amount").alias("total_payment")
    ) \
    .orderBy(
        "c.campaign_id",
        F.desc("request_count")
    )

display(campaign_status_analytics)

**Data Quality Validation Framework**

In [ ]:
# Validate dim_date covers the campaign date range

campaign_range = spark.table("dim_campaigns").agg(
    F.min("start_date").alias("min_campaign_date"),
    F.max("end_date").alias("max_campaign_date")
).collect()[0]

date_range = spark.table("dim_date").agg(
    F.min("date").alias("min_dim_date"),
    F.max("date").alias("max_dim_date")
).collect()[0]

dim_date_valid = (
    date_range["min_dim_date"] <= campaign_range["min_campaign_date"]
    and
    date_range["max_dim_date"] >= campaign_range["max_campaign_date"]
)

print(
    f"dim_date: {'PASS' if dim_date_valid else 'FAIL'} | "
    f"Campaign range = {campaign_range['min_campaign_date']} "
    f"to {campaign_range['max_campaign_date']} | "
    f"Dim date range = {date_range['min_dim_date']} "
    f"to {date_range['max_dim_date']}"
)

In [ ]:
# Step 1: Row Count Validation

expected_counts = {
    "dim_user": 10000,
    "dim_campaigns": 5000,
    "fact_ad_request": 100000
}

for table_name, expected_count in expected_counts.items():
    
    actual_count = spark.table(table_name).count()
    
    if actual_count == expected_count:
        print(f"{table_name}: PASS | Expected = {expected_count}, Actual = {actual_count}")
    else:
        print(f"{table_name}: FAIL | Expected = {expected_count}, Actual = {actual_count}")

In [ ]:
# Step 2: Primary Key Validation
# We want to make sure our dimension tables don't contain duplicate keys.

pk_checks = {
    "dim_user": ("user_id", 0),
    "dim_campaigns": ("campaign_id", 0),
    "dim_date": ("date_id", 0)
}

for table_name, (pk_column, expected_duplicates) in pk_checks.items():

    duplicate_count = (
        spark.table(table_name)
        .groupBy(pk_column)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    if duplicate_count == expected_duplicates:
        print(
            f"{table_name}: PASS | "
            f"Duplicate {pk_column}s = {duplicate_count}"
        )
    else:
        print(
            f"{table_name}: FAIL | "
            f"Duplicate {pk_column}s = {duplicate_count}"
        )

#Passed, This confirms that our dimension keys are unique,
#which is important for a clean Star Schema.

In [ ]:
# Step 3: Foreign Key Validation

fact = spark.table("fact_ad_request")
dim_campaigns = spark.table("dim_campaigns")
dim_user = spark.table("dim_user")
dim_date = spark.table("dim_date")

# Campaign relationship
invalid_campaigns = fact.join(
    dim_campaigns.select("campaign_id").distinct(),
    on="campaign_id",
    how="left_anti"
).count()

# Influencer relationship
invalid_influencers = fact.join(
    dim_user.select("user_id").distinct(),
    fact.influencer_id == dim_user.user_id,
    how="left_anti"
).count()

# Sponsor relationship
invalid_sponsors = fact.join(
    dim_user.select("user_id").distinct(),
    fact.sponsor_id == dim_user.user_id,
    how="left_anti"
).count()

# Date relationship
invalid_dates = fact.join(
    dim_date.select("date_id").distinct(),
    on="date_id",
    how="left_anti"
).count()

print("Campaign relationship:", "PASS" if invalid_campaigns == 0 else "FAIL",
      "| Invalid =", invalid_campaigns)

print("Influencer relationship:", "PASS" if invalid_influencers == 0 else "FAIL",
      "| Invalid =", invalid_influencers)

print("Sponsor relationship:", "PASS" if invalid_sponsors == 0 else "FAIL",
      "| Invalid =", invalid_sponsors)

print("Date relationship:", "PASS" if invalid_dates == 0 else "FAIL",
      "| Invalid =", invalid_dates)

In [ ]:
# Step 4: NULL Foreign Key Validation
#For our Gold fact table, the key columns should not be NULL
#campaign_id,influencer_id,sponsor_id,date_id

null_check = fact.select(
    F.count(F.when(F.col("campaign_id").isNull(), 1)).alias("null_campaign_id"),
    F.count(F.when(F.col("influencer_id").isNull(), 1)).alias("null_influencer_id"),
    F.count(F.when(F.col("sponsor_id").isNull(), 1)).alias("null_sponsor_id"),
    F.count(F.when(F.col("date_id").isNull(), 1)).alias("null_date_id")
)

null_check.show()

In [ ]:
# Step 5: Business Rule Validation
#For our fact_ad_request, the important checks are:
#1. Negative payment amounts,2. Unexpected statuses,3. Missing request IDs

valid_statuses = [
    "pending",
    "accepted",
    "negotiate",
    "completed",
    "paid",
    "new"
]

# Check negative payments
negative_payments = fact.filter(
    F.col("payment_amount") < 0
).count()

# Check unexpected statuses
invalid_statuses = fact.filter(
    F.col("status").isNotNull() &
    ~F.col("status").isin(valid_statuses)
).count()

# Check missing request IDs
missing_request_ids = fact.filter(
    F.col("request_id").isNull()
).count()


print(
    "Negative payments:",
    "PASS" if negative_payments == 0 else "FAIL",
    "| Invalid =", negative_payments
)

print(
    "Invalid statuses:",
    "PASS" if invalid_statuses == 0 else "FAIL",
    "| Invalid =", invalid_statuses
)

print(
    "Missing request IDs:",
    "PASS" if missing_request_ids == 0 else "FAIL",
    "| Missing =", missing_request_ids
)

In [ ]:
# Step 6: Overall Gold Validation Summary
from pyspark.sql import functions as F

validation_results = {
    "Row Count": (
        spark.table("dim_user").count() == 10000
        and spark.table("dim_campaigns").count() == 5000
        and spark.table("dim_date").count() == 670
        and spark.table("fact_ad_request").count() == 98900
    ),

    "Primary Keys": (
        all(
            spark.table(table_name)
            .groupBy(pk_column)
            .count()
            .filter(F.col("count") > 1)
            .count() == 0
            for table_name, pk_column in [
                ("dim_user", "user_id"),
                ("dim_campaigns", "campaign_id"),
                ("dim_date", "date_id")
            ]
        )
    ),

    "Foreign Keys": (
        invalid_campaigns == 0
        and invalid_influencers == 0
        and invalid_sponsors == 0
        and invalid_dates == 0
    ),

    "NULL Foreign Keys": (
        fact.filter(F.col("campaign_id").isNull()).count() == 0
        and fact.filter(F.col("influencer_id").isNull()).count() == 0
        and fact.filter(F.col("sponsor_id").isNull()).count() == 0
        and fact.filter(F.col("date_id").isNull()).count() == 0
    ),

    "Business Rules": (
        negative_payments == 0
        and invalid_statuses == 0
        and missing_request_ids == 0
    )
}

for check, result in validation_results.items():
    print(f"{check}: {'PASS' if result else 'FAIL'}")

print("\nOverall Gold Validation:",
      "PASS" if all(validation_results.values()) else "FAIL")

**Incremental Loading**

In [ ]:
'''
#Step 1 — Establish our current Silver baseline
#this gives last user id, this is incremental loading starting point,
#like a checkpoint
silver_users.select(
    F.max(F.col("id").cast("int")).alias("max_user_id")
).show()
#we need to cast it to integer to get the max value
'''

In [ ]:
'''
#Step 2 — Simulate new incoming data
#we'll create a small incoming DataFrame that represents new raw data,
#that this is new data arriving from the source system
new_users = spark.createDataFrame(
    [
        ("121", "new_user_121", "user121@example.com", "hash121", "influencer", "0"),
        ("122", "new_user_122", "user122@example.com", "hash122", "sponsor", "0"),
        ("123", "new_user_123", "user123@example.com", "hash123", "influencer", "0")
    ],
    [
        "id",
        "username",
        "email",
        "password_hash",
        "role",
        "is_flagged"
    ]
)

display(new_users)
'''

In [ ]:
'''
#Step 3 — Identify only the new records
new_records = new_users.join(
    silver_users,
    on="id",
    how="left_anti"
)

display(new_records)

#I compare incoming records against the existing Delta table and use a left 
#anti join to identify records that are not already present. 
#This allows the pipeline to process only new records instead of reprocessing
#the entire dataset.
'''

In [ ]:
'''
#Step 4 — Add the new records to the Silver Delta table
#using delta merge
from delta.tables import DeltaTable
'''

In [ ]:
'''
#4.1 reference table
#this will ask spark to look for table name silver_users
silver_users_delta = DeltaTable.forName(
    spark,
    "silver_users"
)
'''

In [ ]:
'''
#4.2 merge
silver_users_delta.alias("target") \
    .merge(
        new_records.alias("source"),
        "target.id = source.id"
    ) \
    .whenNotMatchedInsertAll() \
    .execute()

#whenNotMatchedInsertAll means when id not matched, insert the entire new row
'''

In [ ]:
'''
#step 5 - Verify
spark.table("silver_users").count()

spark.table("silver_users") \
    .filter(F.col("id").cast("int") > 120) \
    .orderBy(F.col("id").cast("int")) \
    .show()

#verified
'''

In [ ]:
'''
#Step 6 — Simulate an updated use
silver_users.filter(
    F.col("id") == "50"
).show(truncate=False)
'''

In [ ]:
'''
#Step 6.1 — Create the incoming update
updated_user = spark.createDataFrame(
    [
        (
            "50",
            "gamerguru_50",
            "user50@example.com",
            "scrypt:32768:8:1$gOzWAwnqVmaz1I4v$5b14f834f520e3d232ed3b95bb8fe7c6a13bfa968fa55accfd7e1802a78b3389f03fc1661a4b8d1d2ee843f0d33cc66121a19efcedec6b29d93a9f7c87f5a3f2",
            "sponsor",
            "False"
        )
    ],
    [
        "id",
        "username",
        "email",
        "password_hash",
        "role",
        "is_flagged"
    ]
)

display(updated_user)
'''

In [ ]:
'''
#Step 6.2 — MERGE with UPDATE + INSERT
silver_users_delta.alias("target") \
    .merge(
        updated_user.alias("source"),
        "target.id = source.id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()
'''

In [ ]:
'''
#Step 6.3 — Verify
silver_users.filter(
    F.col("id") == "50"
).show(truncate=False)
'''

In [ ]:
'''
#Step 7 — understanding Delta Lake Time Travel
from delta.tables import DeltaTable

silver_users_delta = DeltaTable.forName(
    spark,
    "silver_users"
)

silver_users_delta.history().select(
    "version",
    "timestamp",
    "operation"
).show(truncate=False)
'''

In [ ]:
'''
#Step 8 — using Time Travel
#old version
users_v8 = spark.read \
    .format("delta") \
    .option("versionAsOf", 8) \
    .table("silver_users")

users_v8.filter(
    F.col("id") == "50"
).select(
    "id",
    "username",
    "role"
).show()

#current verison
silver_users.filter(
    F.col("id") == "50"
).select(
    "id",
    "username",
    "role"
).show()
'''

Commenting out things as this is just for practice and learning, also running
pipeline again to regain the old values

In [ ]:
tables = [
    "dim_user",
    "dim_campaigns",
    "dim_date",
    "fact_ad_request"
]

for table_name in tables:
    df = spark.read.table(table_name)

    output_path = f"Files/PowerBI_Export/{table_name}"

    df.coalesce(1).write \
        .mode("overwrite") \
        .option("header", "true") \
        .csv(output_path)

    print(f"Exported {table_name}: {df.count()} rows")

In [ ]:
tables = [
    "dim_user",
    "dim_campaigns",
    "dim_date",
    "fact_ad_request"
]

for table_name in tables:
    df = spark.read.table(table_name)

    output_path = f"Files/PowerBI_Export/{table_name}"

    df.coalesce(1).write \
        .mode("overwrite") \
        .option("header", "true") \
        .csv(output_path)

    print(f"Exported {table_name}: {df.count()} rows")